# Búsqueda con heurísticas de superadores del globo

Búsquedas acotadas (no exhaustivas) de pares $(G,\{s,t\})$ con $t_2$ estrictamente mayor que $t_2$ del globo, para los mismos $(n, m)$ que `BusquedaSuperadoresNoMaximos`:

1. **Multipartitos**: $G$ cuyo esqueleto ($G$ con todos los puentes colapsados) es un multipartito completo, con todas las ubicaciones de terminales.
2. **Casi-regulares**: $G$ cuyo esqueleto tiene grado máximo y mínimo que difieren en a lo sumo 1, con todas las ubicaciones de terminales.

En ambos casos no se recorren grafos filtrando: se generan directamente los esqueletos de la clase (sin puentes) y se les descolapsan $b$ puentes.

Cada búsqueda persiste todos los superadores (salvo isomorfismo que fija $s,t$) con el mismo formato que `Datos/SuperadoresDelGlobo.json`.

In [ ]:
import os
while not os.path.isdir("DataTypes"):
    os.chdir("..")

import json
import multiprocessing

load("DataTypes/Grafo2Terminales.sage")
load("Scripts/NumeroDe2Bosques.sage")
load("Scripts/GrafoBalloon.sage")
load("Scripts/Esqueleto.sage")

In [ ]:
INICIO, FIN = 7, 10      # rango de n (ambos inclusive) para casi-regulares
INICIO_MULTIPARTITOS, FIN_MULTIPARTITOS = 7, 107   # rango de n (ambos inclusive) para multipartitos
NUM_HILOS = 16           # procesos en paralelo
NUM_PARTES = 4 * NUM_HILOS   # partes de geng (res/mod) por (n, m) en la búsqueda de casi-regulares
ARCHIVO_MULTIPARTITOS = "Datos/SuperadoresMultipartitos.json"
ARCHIVO_CASI_REGULARES = "Datos/SuperadoresCasiRegulares.json"

def ValoresDeM(n):
    return [binomial(n - 2, 2) + 1, binomial(n - 2, 2) + 2]

In [ ]:
def CargarSuperadores(archivo):
    """
        Lee el archivo y retorna {n: {m: {"t2_globo": int, "superadores": [{"s", "t", "t2", "graph6"}, ...]}}}
        con claves enteras. Para reconstruir un grafo:
        Grafo2Terminales(Graph(d["graph6"]), d["s"], d["t"]).
    """
    if not os.path.exists(archivo):
        return {}
    with open(archivo) as f:
        crudo = json.load(f)
    return {int(n): {int(m): d for m, d in por_m.items()} for n, por_m in crudo.items()}

def GuardarSuperadores(datos, archivo):
    os.makedirs(os.path.dirname(archivo), exist_ok=True)
    tmp = archivo + ".tmp"
    with open(tmp, "w") as f:
        json.dump(datos, f, indent=1)
    os.replace(tmp, archivo)   # escritura atómica: una interrupción no deja el JSON truncado

def ClaveCanonica(G, s, t):
    """ Etiqueta canónica del grafo con terminales, salvo isomorfismos que fijan {s,t}. """
    resto = [v for v in G.vertices() if v not in (s, t)]
    return G.canonical_label(partition=[[s, t], resto]).graph6_string()

def EntradaSiSupera(G, s, t, t2_globo):
    """ Retorna la entrada {"s","t","t2","graph6"} si t2 > t2_globo, o None. """
    valor = NumeroDe2Bosques(Grafo2Terminales(G, s, t))
    if valor <= t2_globo:
        return None
    return {"s": int(s), "t": int(t), "t2": int(valor), "graph6": G.graph6_string()}

## Esqueletos y descolapsado de puentes

In [ ]:
def DescolapsarPuente(G):
    """
        Todos los grafos que se obtienen de G al descolapsar un puente: un vértice x se parte
        en x - y. Para que x - y sea puente, cada componente de G - x se queda entera del lado
        de x o entera del lado de y (con todos los vecinos de x que tiene). Al colapsar todos
        los puentes del resultado se recupera el esqueleto de G.
    """
    y = G.order()
    for x in G.vertices():
        resto = Graph(G)
        resto.delete_vertex(x)
        componentes = resto.connected_components()
        for mascara in range(2 ** len(componentes)):
            H = Graph(G)
            H.add_edge(x, y)
            for i, C in enumerate(componentes):
                if (mascara >> i) & 1:
                    for w in C:
                        if G.has_edge(x, w):
                            H.delete_edge(x, w)
                            H.add_edge(y, w)
            yield H

def GrafosConEsqueleto(H, puentes):
    """ Grafos (salvo isomorfismo) con esqueleto H y exactamente `puentes` puentes. """
    actuales = {H.canonical_label().graph6_string(): H}
    for _ in range(puentes):
        siguientes = {}
        for G in actuales.values():
            for K in DescolapsarPuente(G):
                siguientes.setdefault(K.canonical_label().graph6_string(), K)
        actuales = siguientes
    return actuales.values()

def SinPuentes(H):
    return next(iter(H.bridges()), None) is None   # bridges() es un generador

def SuperadoresDeEsqueleto(H, puentes, t2_globo):
    """
        Todos los (G, {s,t}) con t2 > t2_globo, salvo isomorfismo que fija s,t, tales que G tiene
        esqueleto H (sin puentes) y `puentes` puentes. Prueba todas las ubicaciones de terminales.
    """
    encontrados = []
    for G in GrafosConEsqueleto(H, puentes):
        V = G.vertices()
        vistos = set()
        for i in range(G.order()):
            for j in range(i + 1, G.order()):
                entrada = EntradaSiSupera(G, V[i], V[j], t2_globo)
                if entrada is None:
                    continue
                clave = ClaveCanonica(G, V[i], V[j])
                if clave not in vistos:
                    vistos.add(clave)
                    encontrados.append(entrada)
    return encontrados

def PuentesPosibles(n, m):
    """ Cantidades b de puentes compatibles con un esqueleto sin puentes de n - b vértices y m - b aristas. """
    return [b for b in range(n - 2) if binomial(n - b, 2) >= m - b >= n - b]

## 1. Multipartitos (sobre el esqueleto)

In [ ]:
def ParticionesCandidatas(k, aristas):
    """
        Particiones de k en al menos 2 partes cuyo multipartito completo tiene `aristas` aristas
        y no tiene puentes (es decir, no es una estrella).
    """
    return [list(p) for p in Partitions(k)
            if len(p) >= 2 and binomial(k, 2) - sum(binomial(x, 2) for x in p) == aristas
            and SinPuentes(graphs.CompleteMultipartiteGraph(list(p)))]

def SuperadoresMultipartitosParticion(args):
    """ Trabajo de un proceso: el multipartito de `partes` como esqueleto, con `puentes` puentes. """
    partes, puentes, t2_globo = args
    return SuperadoresDeEsqueleto(graphs.CompleteMultipartiteGraph(partes), puentes, t2_globo)

def SuperadoresMultipartitos(n, m, num_hilos=NUM_HILOS):
    """
        Retorna (t2 del globo, lista de TODOS los pares (G, {s,t}) con t2 > t2(globo) cuyo esqueleto
        es multipartito completo, ordenada por t2 decreciente. Esqueletos distintos dan grafos
        distintos, así que no hay duplicados entre procesos.
    """
    t2_globo = int(NumeroDe2Bosques(ConstruirGrafo2TerminalesBalloon(n, m)))
    trabajos = [(partes, b, t2_globo)
                for b in PuentesPosibles(n, m)
                for partes in ParticionesCandidatas(n - b, m - b)]
    with multiprocessing.get_context("fork").Pool(num_hilos) as pool:
        resultados = pool.map(SuperadoresMultipartitosParticion, trabajos, chunksize=1)
    superadores = sorted((d for parte in resultados for d in parte), key=lambda d: -d["t2"])
    return t2_globo, superadores

In [ ]:
def PartesDelEsqueleto(entrada):
    """
        Retorna (partes, puentes): las partes del multipartito completo que es esqueleto del grafo
        de la entrada como lista de pares (tamaño, marca), con marca "s", "t", "st" o "" según
        qué terminales (proyectados al esqueleto) contiene la parte, y la cantidad de puentes del grafo.
        Se ordena por tamaño decreciente y, a igual tamaño, primero las partes con terminales.
    """
    G = Graph(entrada["graph6"])
    TE = ObtenerEsqueleto(Grafo2Terminales(G, entrada["s"], entrada["t"]))
    partes = []
    for C in TE.grafo.complement().connected_components_subgraphs():
        vertices = C.vertices()
        marca = ("s" if TE.s in vertices else "") + ("t" if TE.t in vertices else "")
        partes.append((C.order(), marca))
    partes.sort(key=lambda p: (-p[0], p[1] == "", p[1]))
    return partes, int(G.size() - TE.grafo.size())

def TestigosMaximos(superadores):
    """
        Testigos que maximizan t2 entre los superadores: uno por cada (partes del esqueleto con
        ubicación de terminales, cantidad de puentes) distinto entre los de t2 máximo, con sus
        entradas {"s","t","t2","graph6"}. Lista vacía si no hay superadores.
    """
    if not superadores:
        return []
    agrupados = {}
    for e in superadores:
        if e["t2"] != superadores[0]["t2"]:
            break
        partes, puentes = PartesDelEsqueleto(e)
        agrupados.setdefault((tuple(partes), puentes), []).append(e)
    return [{"partes": [p for p, _ in partes], "marcas": [mk for _, mk in partes],
             "puentes": puentes, "entradas": entradas}
            for (partes, puentes), entradas in agrupados.items()]

def NombreTestigo(testigo):
    """
        K(a,b,c,...) del esqueleto; un sufijo _s, _t o _st marca la parte que contiene
        a los terminales, p. ej. K(3_st,3,3,2,1). Si hay puentes se agregan al final.
    """
    partes = [f"{p}_{mk}" if mk else str(p) for p, mk in zip(testigo["partes"], testigo["marcas"])]
    nombre = "K(" + ",".join(partes) + ")"
    if testigo["puentes"]:
        nombre += f" + {testigo['puentes']} puente(s)"
    return nombre

In [ ]:
multipartitos_por_nm = CargarSuperadores(ARCHIVO_MULTIPARTITOS)
for n in range(INICIO_MULTIPARTITOS, FIN_MULTIPARTITOS + 1):
    for m in ValoresDeM(n):
        t2_globo, superadores = SuperadoresMultipartitos(n, m)
        testigos = TestigosMaximos(superadores)
        mejor = superadores[0]["t2"] if superadores else None
        print(f"n={n}, m={m}: t2(globo) = {t2_globo}, #superadores multipartitos (salvo isomorfismo) = {len(superadores)}, "
              f"mejor t2 = {mejor}, testigo(s) = {', '.join(NombreTestigo(t) for t in testigos) or None}")
        multipartitos_por_nm.setdefault(int(n), {})[int(m)] = {
            "t2_globo": t2_globo, "testigos_maximos": testigos, "superadores": superadores}
        GuardarSuperadores(multipartitos_por_nm, ARCHIVO_MULTIPARTITOS)

n=7, m=11: t2(globo) = 200, #superadores multipartitos (salvo isomorfismo) = 0, mejor t2 = None, testigo(s) = None
n=7, m=12: t2(globo) = 300, #superadores multipartitos (salvo isomorfismo) = 6, mejor t2 = 360, testigo(s) = K(3_st,2,1) + 1 puente(s)
n=8, m=16: t2(globo) = 2160, #superadores multipartitos (salvo isomorfismo) = 5, mejor t2 = 2688, testigo(s) = K(3_st,3,1) + 1 puente(s)
n=8, m=17: t2(globo) = 3024, #superadores multipartitos (salvo isomorfismo) = 5, mejor t2 = 4200, testigo(s) = K(3_st,2,2) + 1 puente(s)
n=9, m=22: t2(globo) = 28812, #superadores multipartitos (salvo isomorfismo) = 16, mejor t2 = 42000, testigo(s) = K(3_st,3,2) + 1 puente(s)
n=9, m=23: t2(globo) = 38416, #superadores multipartitos (salvo isomorfismo) = 12, mejor t2 = 56000, testigo(s) = K(3_st,3,1,1) + 1 puente(s)
n=10, m=29: t2(globo) = 458752, #superadores multipartitos (salvo isomorfismo) = 13, mejor t2 = 694575, testigo(s) = K(4_st,2,2,1) + 1 puente(s)
n=10, m=30: t2(globo) = 589824, #superadores mult

## 2. Casi-regulares (sobre el esqueleto)

In [ ]:
def GrafosEsqueletoCasiRegularParte(n, m, res, mod):
    """
        Parte res/mod de los grafos conexos y sin puentes de n vértices y m aristas con grados en
        {floor(2m/n), ceil(2m/n)} (casi-regulares: max - min <= 1). Son los posibles esqueletos.
    """
    d, D = (2 * m) // n, -((-2 * m) // n)
    for H in graphs.nauty_geng(f"{n} -c -d{d} -D{D} {m}:{m} {res}/{mod}"):
        if SinPuentes(H):
            yield H

def SuperadoresCasiRegularesParte(args):
    """
        Trabajo de un proceso: esqueletos H de la parte res/mod con n - puentes vértices,
        sus expansiones a n vértices y todas las ubicaciones de terminales.
    """
    n, m, puentes, res, mod, t2_globo = args
    encontrados = []
    for H in GrafosEsqueletoCasiRegularParte(n - puentes, m - puentes, res, mod):
        encontrados.extend(SuperadoresDeEsqueleto(H, puentes, t2_globo))
    return encontrados

def SuperadoresCasiRegulares(n, m, num_hilos=NUM_HILOS, num_partes=NUM_PARTES):
    """
        Retorna (t2 del globo, lista de TODOS los pares (G, {s,t}) con esqueleto casi-regular y
        t2 > t2(globo), ordenada por t2 decreciente. Para cada cantidad de puentes b posible
        (esqueleto de n - b vértices y m - b aristas) las partes de geng son disjuntas, y esqueletos
        distintos dan grafos distintos, así que no hay duplicados.
    """
    t2_globo = int(NumeroDe2Bosques(ConstruirGrafo2TerminalesBalloon(n, m)))
    trabajos = [(int(n), int(m), b, res, num_partes, t2_globo)
                for b in PuentesPosibles(n, m)
                for res in range(num_partes)]
    with multiprocessing.get_context("fork").Pool(num_hilos) as pool:
        partes = pool.map(SuperadoresCasiRegularesParte, trabajos, chunksize=1)
    superadores = sorted((d for parte in partes for d in parte), key=lambda d: -d["t2"])
    return t2_globo, superadores

In [ ]:
casi_regulares_por_nm = CargarSuperadores(ARCHIVO_CASI_REGULARES)
for n in range(INICIO, FIN + 1):
    for m in ValoresDeM(n):
        t2_globo, superadores = SuperadoresCasiRegulares(n, m)
        mejor = superadores[0]["t2"] if superadores else None
        print(f"n={n}, m={m}: t2(globo) = {t2_globo}, #superadores casi-regulares (salvo isomorfismo) = {len(superadores)}, "
              f"mejor t2 = {mejor}")
        casi_regulares_por_nm.setdefault(int(n), {})[int(m)] = {"t2_globo": t2_globo, "superadores": superadores}
        GuardarSuperadores(casi_regulares_por_nm, ARCHIVO_CASI_REGULARES)